In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT   # 가장 좋은 사전학습 가중치
print(weights.transforms())          # 이 모델이 원하는 사진 손질법

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [2]:
from torchvision import transforms as T
from torchvision.datasets import OxfordIIITPet

MEAN = [0.485, 0.456, 0.406]   # 2-1에서 메모한 숫자
STD = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),   # 무작위로 오려서 정사각형으로
    T.RandomHorizontalFlip(p=0.5),                 # 반반 확률로 좌우 뒤집기
    T.ToTensor(),                                   # 사진 → 숫자 상자(텐서)
    T.Normalize(MEAN, STD),                         # 색 맞추기
])

test_tf = T.Compose([
    T.Resize(256),                                  # 짧은 쪽을 줄이기
    T.CenterCrop(224),                              # 가운데 오리기
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

In [3]:
sample = OxfordIIITPet(root="../data", split="trainval",
                       target_types="binary-category", transform=train_tf)
x, label = sample[0]
assert x.shape == (3, 224, 224), f"(3, 224, 224)여야 해요. 지금: {tuple(x.shape)}"
print("통과! 사진 한 장 →", tuple(x.shape))

통과! 사진 한 장 → (3, 224, 224)


In [4]:
from torch.utils.data import Subset

full_train = OxfordIIITPet(root="../data", split="trainval", target_types="binary-category", transform=train_tf)
full_val = OxfordIIITPet(root="../data", split="trainval", target_types="binary-category", transform=test_tf)
test_set = OxfordIIITPet(root="../data", split="test", target_types="binary-category", transform=test_tf)

g = torch.Generator().manual_seed(42)                 # 매번 똑같이 섞이도록 고정
idx = torch.randperm(len(full_train), generator=g).tolist()   # 사진 번호 섞기
n_train = int(len(idx) * 0.8)                        # 연습 문제 개수

train_set = Subset(full_train, idx[:n_train])         # 앞부분
val_set = Subset(full_val, idx[n_train:])             # 나머지

print("연습:", len(train_set), "/ 모의고사:", len(val_set), "/ 수능:", len(test_set))

연습: 2944 / 모의고사: 736 / 수능: 3669


In [5]:
assert len(train_set) == 2944 and len(val_set) == 736, "2944 / 736이 나와야 해요"
assert not set(train_set.indices) & set(val_set.indices), "연습과 모의고사에 같은 사진이 있으면 안 돼요"
print("통과!")

통과!


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))     # 한 묶음만 꺼내 보기
print(xb.shape, yb[:8])

torch.Size([32, 3, 224, 224]) tensor([0, 1, 0, 1, 0, 1, 1, 0])


In [7]:
assert xb.shape == (32, 3, 224, 224), f"지금: {tuple(xb.shape)}"
print("통과! 사진 32장 묶음 →", tuple(xb.shape))

통과! 사진 32장 묶음 → (32, 3, 224, 224)


In [8]:
model = resnet18(weights=weights).eval()      # 사전학습 모델 불러오기 ("지금은 시험 시간")
imagenet_names = weights.meta["categories"]   # ImageNet 1000종류 이름표

raw_test = OxfordIIITPet(root="../data", split="test", target_types="binary-category")   # 손질 전 사진
for i in [0, 1000, 2000]:
    img, label = raw_test[i]
    x = weights.transforms()(img).unsqueeze(0)   # 손질 + 한 장짜리 묶음으로 (1, 3, 224, 224)
    with torch.no_grad():
        prob = model(x).softmax(dim=1)[0]        # 1000종류 각각의 확률
    top = prob.topk(3)                        # 확률 높은 순서로 3개
    guesses = [f"{imagenet_names[j]} {p:.0%}" for p, j in zip(top.values, top.indices)]
    print(["고양이", "개"][label], "→", guesses)

고양이 → ['Egyptian cat 70%', 'tabby 8%', 'lynx 5%']
개 → ['Chihuahua 98%', 'French bulldog 1%', 'muzzle 0%']
고양이 → ['tabby 53%', 'lynx 18%', 'tiger cat 17%']


In [9]:
print("바꾸기 전:", model.fc)

model.fc = nn.Linear(512, 2)    # 새 마지막 층 (들어오는 수, 나가는 수)

print("바꾼 후:", model.fc)

바꾸기 전: Linear(in_features=512, out_features=1000, bias=True)
바꾼 후: Linear(in_features=512, out_features=2, bias=True)


In [10]:
out = model(xb[:4])                  # 4-2에서 꺼낸 사진 중 4장
assert out.shape == (4, 2), f"사진마다 점수 2개가 나와야 해요. 지금: {tuple(out.shape)}"
print("통과! 이제 개/고양이 두 가지로만 대답해요", tuple(out.shape))

통과! 이제 개/고양이 두 가지로만 대답해요 (4, 2)
